In following the hypothesis from the previous logistic regressor, I want to try adding interaction terms between players to better approximate a team outcome. However, engineering up to a degree 5 polynomial out of a feature set with 83 features to begin, it will have way too many dimensions and become too big to store. As such, I will only add the degree 5 interaction terms of the players by stat on that team then do forward step-size selection.

I know I have to do a subset of the interaction terms because I tried to do a full degree 5 polynomial expansion on all columns. The array this would yield turned out to be ~10 million collumns wide with 5500 entries, needing 614 GiB of RAM to exist in memory. Obviously, we cannot have this. Additionally, we really only expect how the 5 players interact together by stat to have any real predictive power logically.

I know we have to do a subset of the interaction terms because I tried to do a full degree 5 polynomial feature expansion which yielded an array with ~10 million columns which needed 614 GB of ram which is highly impractical

In [57]:
import pandas as pd
df = pd.read_csv("../../data/rolling_feature_engineered_tier_1_games.csv")
df.shape

(5539, 84)

In [58]:
df.columns

Index(['previous_10_game_team1_player1_average_kills',
       'previous_10_game_team1_player1_average_deaths',
       'previous_10_game_team1_player1_average_assists',
       'previous_10_game_team1_player1_average_adr',
       'previous_10_game_team1_player1_average_kast',
       'previous_10_game_team1_player1_average_kddiff',
       'previous_10_game_team1_player2_average_kills',
       'previous_10_game_team1_player2_average_deaths',
       'previous_10_game_team1_player2_average_assists',
       'previous_10_game_team1_player2_average_adr',
       'previous_10_game_team1_player2_average_kast',
       'previous_10_game_team1_player2_average_kddiff',
       'previous_10_game_team1_player3_average_kills',
       'previous_10_game_team1_player3_average_deaths',
       'previous_10_game_team1_player3_average_assists',
       'previous_10_game_team1_player3_average_adr',
       'previous_10_game_team1_player3_average_kast',
       'previous_10_game_team1_player3_average_kddiff',
       

Let's drop arbitrary information like ids, names, etc.

In [59]:
to_drop = []
to_drop.extend([f"team{i}_player{k}_id" for i in range(1,3) for k in range(1,6)])
to_drop.extend([f"team{i}_id" for i in range(1,3)])
to_drop.extend([f"team{i}" for i in range(1,3)])
to_drop.extend(["tournament", "match_id", "game_id", "map_id", "map_name", "datetime"])
df = df.drop(to_drop, axis=1)
df.columns

Index(['previous_10_game_team1_player1_average_kills',
       'previous_10_game_team1_player1_average_deaths',
       'previous_10_game_team1_player1_average_assists',
       'previous_10_game_team1_player1_average_adr',
       'previous_10_game_team1_player1_average_kast',
       'previous_10_game_team1_player1_average_kddiff',
       'previous_10_game_team1_player2_average_kills',
       'previous_10_game_team1_player2_average_deaths',
       'previous_10_game_team1_player2_average_assists',
       'previous_10_game_team1_player2_average_adr',
       'previous_10_game_team1_player2_average_kast',
       'previous_10_game_team1_player2_average_kddiff',
       'previous_10_game_team1_player3_average_kills',
       'previous_10_game_team1_player3_average_deaths',
       'previous_10_game_team1_player3_average_assists',
       'previous_10_game_team1_player3_average_adr',
       'previous_10_game_team1_player3_average_kast',
       'previous_10_game_team1_player3_average_kddiff',
       

Before doing the feature engineering, let's drop the categorical features then add them back in later since it doesn't make much sense to engineer with them since they either multiply by a 0 or 1, not modifying anything meaningfully.

In [60]:
y = df.team1_win
X = df.drop("team1_win", axis=1)
X_cat = df.bestOf
X_numerical = X.drop("bestOf", axis=1)

In [61]:
X_numerical.shape

(5539, 62)

In [62]:
stat_sets = {}
stats = ["kills", "deaths", "assists", "adr", "kast", "kddiff"]
for i in range(1, 3):
    for stat in stats:
        set = []
        for k in range(1, 6):
            set.append(f"previous_10_game_team{i}_player{k}_average_{stat}")
        current = stat_sets.get(f"team{i}", {})
        current[stat] = set
        stat_sets[f"team{i}"] = current
            
#Generate the 5 player interaction terms by stat
for team in stat_sets.keys():
    for stat in stat_sets[team]:
        #Asked ChatGPT how to multiply the values of a given set of columns together
        X_numerical[f"{team}_{stat}_interaction"] = X_numerical[stat_sets[team][stat]].prod(axis=1)

In [63]:
X_numerical.shape

(5539, 74)

In [64]:
X_numerical.columns

Index(['previous_10_game_team1_player1_average_kills',
       'previous_10_game_team1_player1_average_deaths',
       'previous_10_game_team1_player1_average_assists',
       'previous_10_game_team1_player1_average_adr',
       'previous_10_game_team1_player1_average_kast',
       'previous_10_game_team1_player1_average_kddiff',
       'previous_10_game_team1_player2_average_kills',
       'previous_10_game_team1_player2_average_deaths',
       'previous_10_game_team1_player2_average_assists',
       'previous_10_game_team1_player2_average_adr',
       'previous_10_game_team1_player2_average_kast',
       'previous_10_game_team1_player2_average_kddiff',
       'previous_10_game_team1_player3_average_kills',
       'previous_10_game_team1_player3_average_deaths',
       'previous_10_game_team1_player3_average_assists',
       'previous_10_game_team1_player3_average_adr',
       'previous_10_game_team1_player3_average_kast',
       'previous_10_game_team1_player3_average_kddiff',
       

Now, let's one-hot encode and put everything back together.

In [65]:
X_cat = pd.get_dummies(X_cat, prefix="bestOf", drop_first=True, dtype=int)
X = pd.concat([X_numerical, X_cat], axis=1)
X.sample(5)
X.columns

Index(['previous_10_game_team1_player1_average_kills',
       'previous_10_game_team1_player1_average_deaths',
       'previous_10_game_team1_player1_average_assists',
       'previous_10_game_team1_player1_average_adr',
       'previous_10_game_team1_player1_average_kast',
       'previous_10_game_team1_player1_average_kddiff',
       'previous_10_game_team1_player2_average_kills',
       'previous_10_game_team1_player2_average_deaths',
       'previous_10_game_team1_player2_average_assists',
       'previous_10_game_team1_player2_average_adr',
       'previous_10_game_team1_player2_average_kast',
       'previous_10_game_team1_player2_average_kddiff',
       'previous_10_game_team1_player3_average_kills',
       'previous_10_game_team1_player3_average_deaths',
       'previous_10_game_team1_player3_average_assists',
       'previous_10_game_team1_player3_average_adr',
       'previous_10_game_team1_player3_average_kast',
       'previous_10_game_team1_player3_average_kddiff',
       

Now, let's do our train/test split before doing forward feature selection

In [66]:
import math
split_point = math.ceil(len(X) * 0.8)
# SPLIT MUST BE TEMPORAL TO AVOID TEST LEAKAGE INTO TRAIN SINCE
# FEATURES ARE CUMULATIVE
X_train = X.iloc[:split_point] # 0 to split_point - 1
X_test = X.iloc[split_point:] # split_point to len(df)
y_train = y.iloc[:split_point]
y_test = y.iloc[split_point:]

Let's standardize

In [67]:
from sklearn.preprocessing import StandardScaler
stnd = StandardScaler().set_output(transform="pandas")
X_train = stnd.fit_transform(X_train)
X_test = stnd.transform(X_test)

In [68]:
X_train["bias"] = 1
X_test["bias"] = 1

In [69]:
#From Homework 6
from sklearn.model_selection import TimeSeriesSplit, cross_validate # we must use this as our cross validation split generator since our data is time dependent
from sklearn.linear_model import LogisticRegression
def SelectFeature(model, candidates, X, y):
    best_R2 = None
    best_R2_feature = None
    for candidate in candidates:
        #Get the slice of df consisting the new model with a feature added
        new_model = model.copy() # so it doesn't mutate the input array
        #Asked ChatGPT how to get a slice of columns given col labels array
        new_model.append(candidate)
        X_temp = X[new_model]
        #Get the CV R2
        lr = LogisticRegression(fit_intercept=False) # bias is one of our features explicitly
        cv = cross_validate(lr, X_temp, y, n_jobs=-1, cv=TimeSeriesSplit())
        test_r2 = cv["test_score"].mean()
        if(best_R2 is None):
            best_R2 = test_r2
            best_R2_feature = candidate
            continue
        if(best_R2 < test_r2):
            best_R2 = test_r2
            best_R2_feature = candidate
    return (best_R2_feature, best_R2)

In [70]:
models = []
R2s = []
model = ["bias"] #initialize with bias
#Score the bias only model
models.append(model.copy())
lr = LogisticRegression(fit_intercept=False)
result = cross_validate(lr, X_train[model], y_train, n_jobs=-1, cv=TimeSeriesSplit())
R2s.append(result["test_score"].mean())
#Ask ChatGPT how to make .columns a mutable list
candidates = list(X_train.drop("bias",axis=1).columns)
while len(candidates) > 0:
    best_feature, best_r2 = SelectFeature(model,candidates,X_train,y_train)
    model.append(best_feature)
    R2s.append(best_r2)
    models.append(model.copy())
    candidates.remove(best_feature)

In [74]:
model_df = pd.DataFrame()
model_df["Model"] = models
model_df["Validation R2"] = R2s
model_df = model_df.sort_values(by="Validation R2", ascending=False)
model_df

,Model,Validation R2
7,"[bias, team1_adr_interaction, previous_10_game...",0.582114
32,"[bias, team1_adr_interaction, previous_10_game...",0.582114
19,"[bias, team1_adr_interaction, previous_10_game...",0.581843
9,"[bias, team1_adr_interaction, previous_10_game...",0.581572
31,"[bias, team1_adr_interaction, previous_10_game...",0.581572
...,...,...
1,"[bias, team1_adr_interaction]",0.559892
74,"[bias, team1_adr_interaction, previous_10_game...",0.559892
75,"[bias, team1_adr_interaction, previous_10_game...",0.555285
76,"[bias, team1_adr_interaction, previous_10_game...",0.551491


In [79]:
best_model = model_df.iloc[0]["Model"]
best_model

['bias',
 'team1_adr_interaction',
 'previous_10_game_team2_player2_average_kast',
 'previous_10_game_team1_player4_average_kddiff',
 'previous_10_game_team2_player4_average_kills',
 'previous_10_game_team2_player5_average_assists',
 'previous_10_game_team2_player1_average_assists',
 'team2_kills_interaction']

I am surprised that only two interaction features made it in.

In [80]:
lr = LogisticRegression(fit_intercept=False)
lr.fit(X_train[best_model], y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",False
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mu

How did it assign weights?

In [84]:
weights = pd.DataFrame()
weights["Feature"] = best_model
weights["Weight"] = lr.coef_[0]
weights = weights.sort_values(by="Weight", ascending = False)
weights.head(10)

,Feature,Weight
0,bias,0.189847
1,team1_adr_interaction,0.188166
3,previous_10_game_team1_player4_average_kddiff,0.052120
5,previous_10_game_team2_player5_average_assists,0.036472
7,team2_kills_interaction,-0.013234
6,previous_10_game_team2_player1_average_assists,-0.014357
2,previous_10_game_team2_player2_average_kast,-0.087856
4,previous_10_game_team2_player4_average_kills,-0.126324


It is most valuing the team1_adr_interaction. It also gives a large negative weight to team 2 player 4 average kills, but again this is likely modeling noise because player is not consistently the highest skill player. It did identify that team2_kills_interaction has a negative weight, but not by much. It would seem that this time all the negative weights do at least correspond to team 2 statistics, which makes more sense than last time.

In [81]:
lr.score(X_train[best_model], y_train)

0.5767148014440433

In [86]:
((y_train == y_train.mode()[0]).sum()) / len(y_train)

np.float64(0.5464801444043321)

In [82]:
lr.score(X_test[best_model], y_test)

0.5465221318879856

In [85]:
((y_test == y_test.mode()[0]).sum()) / len(y_test)

np.float64(0.5582655826558266)

This model performs ~1% worse on the test set and ~3% better on the train set. This indicates to me that the model is capturing mostly noise such as player skill shuffling in the players 1-5 slots, since most of the weights in the ideal model were individual player x statistics again. As a final resort on trying to get logistic regression to work, I think it is worth it to remove the individual player statistics altogether and compound them into one team stat to remove the modeling of noise of the player position shuffling.